In [15]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import sqlite3
import gradio as gr

In [3]:
load_dotenv(override=True)

google_api_key = os.getenv('GEMINI_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
    
MODEL = "gemini-2.5-flash"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [16]:
conn = sqlite3.connect("food.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS food (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    price INTEGER,
    veg INTEGER
)
""")

conn.commit()
conn.close()

In [13]:
system_message = """
You are a helpful assistant for a North Indian food restaurant called Uttam-Rasoi
Welcome the customer in a great way, making him know how good our chat and restaurant is, giving wide variety or North Indian food
Always be accurate. If you don't know the answer, say so.
"""

In [59]:
conn = sqlite3.connect("food.db")
cursor = conn.cursor()

foods = [
    ("Paneer Butter Masala", 250, 1),
    ("Chicken Biryani", 320, 0),
    ("Veg Fried Rice", 180, 1),
    ("Rajma Chawal", 180, 1),
    ("Chhole Bhature", 60, 1),
    ("Fish Curry", 60, 0)
]

cursor.executemany(
    "INSERT INTO food (name, price, veg) VALUES (?, ?, ?)",
    foods
)

conn.commit()
conn.close()

In [35]:
DB = "food.db"

def get_food_price(name):
    print(f"DATABASE TOOL CALLED: Getting price for {name}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM food WHERE LOWER(name) = ?",
            (name.lower(),)
        )
        result = cursor.fetchone()

    return (
        f"Price of {name} is ₹{result[0]}"
        if result
        else "We do not have this item on our menu, extremely sorry for that"
    )

In [ ]:
get_food_price("Chhole Bhature")

In [40]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_food_price",
            "description": "Get the price of a food item from the menu",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Name of the food item"
                    }
                },
                "required": ["name"]
            }
        }
    }
]

In [60]:
def list_menu(veg=None):
    """
    veg = 1  → veg items
    veg = 0  → non-veg items
    veg = None → full menu
    """
    import sqlite3

    with sqlite3.connect("food.db") as conn:
        cur = conn.cursor()

        if veg is None:
            cur.execute("SELECT name, price, veg FROM food")
        else:
            cur.execute("SELECT name, price, veg FROM food WHERE veg = ?", (veg,))

        rows = cur.fetchall()

    if not rows:
        return "No items found."

    lines = []
    for name, price, veg_flag in rows:
        tag = "🟢 Veg" if veg_flag == 1 else "🔴 Non-Veg"
        lines.append(f"- **{name}** — ₹{price} ({tag})")

    return "\n".join(lines)


In [61]:
{
    "type": "function",
    "function": {
        "name": "list_menu",
        "description": "List menu items by veg or non-veg category",
        "parameters": {
            "type": "object",
            "properties": {
                "veg": {
                    "type": ["integer", "null"],
                    "description": "1 for veg, 0 for non-veg, null for full menu"
                }
            }
        }
    }
}


{'type': 'function',
 'function': {'name': 'list_menu',
  'description': 'List menu items by veg or non-veg category',
  'parameters': {'type': 'object',
   'properties': {'veg': {'type': ['integer', 'null'],
     'description': '1 for veg, 0 for non-veg, null for full menu'}}}}}

In [62]:
def handle_tool_calls(message):
    import json

    responses = []

    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "get_food_price":
            name = arguments.get("name")
            result = get_food_price(name)
        elif tool_call.function.name == "list_menu":
            veg = arguments.get("veg")
            result = list_menu(veg)
        else:
            result = "Unknown tool call."
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses


In [55]:
from rapidfuzz import process, fuzz


def get_all_dish_names():
    with sqlite3.connect(DB) as conn:
        cur = conn.cursor()
        cur.execute("SELECT name FROM food")
        return [row[0] for row in cur.fetchall()]

def resolve_dish_name_with_fuzzy(message, threshold=75):
    menu_items = get_all_dish_names()
    for dish in menu_items:
        if dish.lower() in message.lower():
            return dish, "exact"
    match, score, _ = process.extractOne(
        message,
        menu_items,
        scorer=fuzz.token_sort_ratio
    )

    if score >= threshold:
        return match, "fuzzy"

    return None, None

In [64]:
def detect_menu_intent(message):
    msg = message.lower()

    if "non veg" in msg or "non-veg" in msg:
        return "nonveg"

    if "veg" in msg:
        return "veg"

    if "menu" in msg or "all items" in msg:
        return "all"

    return None


In [65]:
def chat(message, history):

    menu_intent = detect_menu_intent(message)

    if menu_intent == "nonveg":
        return list_menu(veg=0)

    if menu_intent == "veg":
        return list_menu(veg=1)

    if menu_intent == "all":
        return list_menu(veg=None)

    dish, match_type = resolve_dish_name_with_fuzzy(message)
    if match_type == "fuzzy":
        return f"Did you mean **{dish}**?"
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = gemini.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

        response = gemini.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat).launch()